# Optimize σx with Jx(t), Δ(t) Fourier (black-box SciPy)


In [3]:
import numpy as np
from scipy.linalg import lu_factor, lu_solve

# ---- Pauli operators + basic utilities ----
I2 = np.array([[1., 0.], [0., 1.]], dtype=complex)
sx = np.array([[0., 1.], [1., 0.]], dtype=complex)
sy = np.array([[0., -1j], [1j, 0.]], dtype=complex)
sz = np.array([[1., 0.], [0., -1.]], dtype=complex)
sigma_minus = np.array([[0., 1.], [0., 0.]], dtype=complex)  # |0><1|

def vec(A: np.ndarray) -> np.ndarray:
    """Column-stacking vec (Fortran order)."""
    return np.asarray(A, dtype=complex).reshape(-1, order="F")

def mat(v: np.ndarray) -> np.ndarray:
    """Inverse of vec for 2x2 matrices."""
    return np.asarray(v, dtype=complex).reshape((2, 2), order="F")

def trace_vec() -> np.ndarray:
    """Vector such that Tr(rho) = trace_vec()^T vec(rho)."""
    return vec(I2)

def random_density(rng: np.random.Generator) -> np.ndarray:
    """Random physical single-qubit density matrix (Hilbert-Schmidt)."""
    A = rng.standard_normal((2, 2)) + 1j * rng.standard_normal((2, 2))
    rho = A @ A.conj().T
    rho = rho / np.trace(rho)
    return rho

# ---- Fourier basis over normalized time s=t/T ----
def basis_matrix(s_nodes: np.ndarray, M: int) -> np.ndarray:
    """B[j,:]=[cos(2π*1*s_j)..cos(2π*M*s_j), sin(2π*1*s_j)..sin(2π*M*s_j)]."""
    m = np.arange(1, M + 1).reshape(1, -1)
    ang = 2 * np.pi * s_nodes.reshape(-1, 1) * m
    C = np.cos(ang)
    S = np.sin(ang)
    return np.hstack([C, S])  # (N, 2M)

# ---- NESS fixed-point solve (works for any 4x4 Floquet map Phi you build) ----
def ness_augmented_system(Phi: np.ndarray):
    """
    Solve r* = Phi r* with Tr(rho)=1 using an augmented linear system:
      [I-Phi, c] [r*]   [0]
      [c^T , 0] [λ ] = [1]
    where c = vec(I).
    """
    n = Phi.shape[0]
    c = trace_vec().reshape(-1, 1)
    A = np.block([
        [np.eye(n, dtype=complex) - Phi, c],
        [c.T, np.zeros((1, 1), dtype=complex)]
    ])
    b = np.zeros((n + 1,), dtype=complex)
    b[-1] = 1.0
    return A, b

In [4]:
import os, time, json
import numpy as np
from scipy.optimize import minimize
from pathlib import Path

import qutip as qt
from scipy.linalg import lu_factor, lu_solve


def sanitize_for_json(obj):
    import numpy as _np
    if isinstance(obj, _np.ndarray):
        return obj.tolist()
    if isinstance(obj, (_np.integer, _np.floating, _np.bool_)):
        return obj.item()
    if isinstance(obj, (_np.complexfloating, complex)):
        c = complex(obj)
        return [float(c.real), float(c.imag)]
    if isinstance(obj, dict):
        return {k: sanitize_for_json(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [sanitize_for_json(v) for v in obj]
    return obj


# =========================
# Parameters (as requested)
# =========================
gamma_down = 4.7
gamma_phi  = 0.3

# Keep N only as a SAMPLING parameter for plots/micromotion markers (NOT for propagators)
N = 48
M = 3

thetaJ_bounds = 50.0
thetaD_bounds = 50.0
T_bounds = (0.05, 2.0)

seed = 1
n_restarts = 10
maxiter = 250
fd_eps = 1e-6

rng = np.random.default_rng(seed)

p = 2 * M
bounds = (
    [(-thetaJ_bounds, thetaJ_bounds)] * p +
    [(-thetaD_bounds, thetaD_bounds)] * p +
    [T_bounds]
)

# =========================
# QuTiP setup
# =========================
# Assumes sx, sy, sz, sigma_minus, vec, mat, basis_matrix, ness_augmented_system, random_density
# are defined in your first "core" cell.

sx_q = qt.Qobj(sx)
sy_q = qt.Qobj(sy)
sz_q = qt.Qobj(sz)
sm_q = qt.Qobj(sigma_minus)

c_ops = []
if gamma_down != 0.0:
    c_ops.append(np.sqrt(gamma_down) * sm_q)
if gamma_phi != 0.0:
    c_ops.append(np.sqrt(gamma_phi) * sz_q)

# QuTiP >=5 prefers dict options (removes FutureWarning)
opts = {"atol": 1e-9, "rtol": 1e-9, "nsteps": 20000}

# Operator basis consistent with vec() (column-stacking):
# These are matrix units E_ij; vec(E_00)=e1, vec(E_10)=e2, vec(E_01)=e3, vec(E_11)=e4
E00 = qt.Qobj(np.array([[1,0],[0,0]], dtype=complex))
E10 = qt.Qobj(np.array([[0,0],[1,0]], dtype=complex))
E01 = qt.Qobj(np.array([[0,1],[0,0]], dtype=complex))
E11 = qt.Qobj(np.array([[0,0],[0,1]], dtype=complex))
BASIS_RHOS = [E00, E10, E01, E11]


def _fourier_no_dc(t: float, args: dict, key: str) -> float:
    """Fourier series with no DC term, evaluated at normalized phase s=t/T."""
    T = float(args["T"])
    s = (t / T) % 1.0
    th = args[key]
    Mloc = int(args["M"])
    val = 0.0
    for m in range(1, Mloc + 1):
        a = float(th[2*m - 2])
        b = float(th[2*m - 1])
        val += a*np.cos(2*np.pi*m*s) + b*np.sin(2*np.pi*m*s)
    return float(val)


def Jx_of_t(t: float, args: dict) -> float:
    return _fourier_no_dc(t, args, "thetaJ")


def D_of_t(t: float, args: dict) -> float:
    return _fourier_no_dc(t, args, "thetaD")


# Time-dependent Hamiltonian for mesolve:
# H(t) = 0.5*Jx(t)*sx + 0.5*Delta(t)*sz
H_td = [
    [0.5 * sx_q, Jx_of_t],
    [0.5 * sz_q, D_of_t],
]


def floquet_map_mesolve_sigmaX(thetaJ: np.ndarray, thetaD: np.ndarray, T: float) -> np.ndarray:
    """
    One-period superoperator Phi (4x4), built by mesolve evolution of 4 basis operators.
    Because BASIS_RHOS are matrix units consistent with vec(), Phi = [vec(rhoT_k)] as columns.
    """
    args = {
        "thetaJ": np.asarray(thetaJ, dtype=float),
        "thetaD": np.asarray(thetaD, dtype=float),
        "T": float(T),
        "M": int(M),
    }
    tlist = [0.0, float(T)]

    cols = []
    for rho0 in BASIS_RHOS:
        res = qt.mesolve(H_td, rho0, tlist, c_ops=c_ops, e_ops=None, args=args, options=opts)
        rhoT = res.states[-1].full()
        cols.append(vec(rhoT))
    Phi = np.column_stack(cols)
    return Phi


def stroboscopic_ness_from_Phi(Phi: np.ndarray) -> np.ndarray:
    """Solve augmented fixed-point system to get r_star (vec(rho_star))."""
    A, b = ness_augmented_system(Phi)
    LU = lu_factor(A)
    x = lu_solve(LU, b)
    r_star = x[:4]
    return r_star


def stroboscopic_value_raw_sigmaX(thetaJ: np.ndarray, thetaD: np.ndarray, T: float) -> complex:
    """Return <sx>_NESS from mesolve-built Phi."""
    Phi = floquet_map_mesolve_sigmaX(thetaJ, thetaD, T)
    r_star = stroboscopic_ness_from_Phi(Phi)
    return (vec(sx).conj().T @ r_star).item()


def fun(x: np.ndarray) -> float:
    """SciPy objective: minimize -<sx> (maximize <sx>)."""
    thetaJ = x[:p]
    thetaD = x[p:2*p]
    T = float(x[2*p])
    sx_val = float(np.real(stroboscopic_value_raw_sigmaX(thetaJ, thetaD, T)))
    return -sx_val


# =========================
# Multistart optimization
# =========================
best = None
t_all = time.time()

for r in range(n_restarts):
    thetaJ0 = 0.2 * thetaJ_bounds * rng.standard_normal(p)
    thetaD0 = 0.2 * thetaD_bounds * rng.standard_normal(p)
    T0 = rng.uniform(T_bounds[0], T_bounds[1])
    x0 = np.concatenate([thetaJ0, thetaD0, [T0]])

    hist = {"sx": [], "T": [], "obj": []}

    def cb(xk):
        thJ = xk[:p]
        thD = xk[p:2*p]
        T_k = float(xk[2*p])
        sx_k = float(np.real(stroboscopic_value_raw_sigmaX(thJ, thD, T_k)))
        hist["sx"].append(sx_k)
        hist["T"].append(T_k)
        hist["obj"].append(-sx_k)

    t0 = time.time()
    res = minimize(
        fun,
        x0,
        method="L-BFGS-B",
        bounds=bounds,
        options={"maxiter": maxiter, "ftol": 1e-12, "eps": fd_eps},
        callback=cb,
    )
    runtime_s = time.time() - t0

    thetaJ_opt = res.x[:p].astype(float)
    thetaD_opt = res.x[p:2*p].astype(float)
    T_opt = float(res.x[2*p])

    sx_opt = float(np.real(stroboscopic_value_raw_sigmaX(thetaJ_opt, thetaD_opt, T_opt)))
    obj_opt = float(res.fun)

    out_r = {
        "restart": r,
        "success": bool(res.success),
        "message": str(res.message),
        "nit": int(res.nit),
        "nfev": int(res.nfev),
        "runtime_s": float(runtime_s),
        "thetaJ": thetaJ_opt,
        "thetaD": thetaD_opt,
        "T": T_opt,
        "sx": sx_opt,
        "obj": obj_opt,
        "history": hist,
    }

    if (best is None) or (out_r["obj"] < best["obj"]):
        best = out_r

    print(f"[restart {r+1}/{n_restarts}] obj={out_r['obj']:.6g}, sx={out_r['sx']:.6g}, T={out_r['T']:.6g}, nit={out_r['nit']}")

total_runtime_s = time.time() - t_all

# =========================
# Save results + diagnostics
# =========================
timestamp = time.strftime("%Y-%m-%d_%H%M%S")
tag = f"{timestamp}_sigmaX_JxDelta_optT_N{N}_M{M}"

run_dir = Path("results") / "runs"
run_dir.mkdir(parents=True, exist_ok=True)

meta = {
    "timestamp": timestamp,
    "tag": tag,
    "objective": "maximize <sigma_x> in stroboscopic Floquet NESS (minimize obj=-<sigma_x>)",
    "controls": "Jx(t) Fourier (optimized) and Delta(t) Fourier (optimized)",
    "method": "L-BFGS-B (black-box FD gradients) + QuTiP mesolve for one-period map construction",
    "gamma_down": gamma_down,
    "gamma_phi": gamma_phi,
    "N": N,          # sampling density for plots/micromotion markers only
    "M": M,
    "thetaJ_bounds": thetaJ_bounds,
    "thetaD_bounds": thetaD_bounds,
    "T_bounds": list(T_bounds),
    "seed": seed,
    "n_restarts": n_restarts,
    "maxiter": maxiter,
    "fd_eps": fd_eps,
    "total_runtime_s": float(total_runtime_s),
}

# Best values (mesolve only)
sx_best = float(best["sx"])
best["sx_mesolve"] = sx_best
best["obj_mesolve"] = float(best["obj"])

out = {**meta, **best}

# ---- Precompute plotting arrays (mesolve-based) ----
thetaJ_best = np.array(best["thetaJ"], dtype=float)
thetaD_best = np.array(best["thetaD"], dtype=float)
T_best = float(best["T"])

Phi_best = floquet_map_mesolve_sigmaX(thetaJ_best, thetaD_best, T_best)
r_star_best = stroboscopic_ness_from_Phi(Phi_best)

sx_star = float(np.real((vec(sx).conj().T @ r_star_best).item()))
sy_star = float(np.real((vec(sy).conj().T @ r_star_best).item()))
sz_star = float(np.real((vec(sz).conj().T @ r_star_best).item()))

# Smooth controls over one period
nplot = 2000
t1 = np.linspace(0.0, T_best, nplot)
s1 = (t1 / T_best) % 1.0
B1 = basis_matrix(s1, M)
Jx_t = np.real(B1 @ thetaJ_best)
D_t  = np.real(B1 @ thetaD_best)

# Optional midpoint sampling for visualization (NOT used for physics)
s_nodes = (np.arange(N) + 0.5) / N
t_s = (np.arange(N) + 0.5) * (T_best / N)
Bmid = basis_matrix(s_nodes, M)
Jx_slices = np.real(Bmid @ thetaJ_best)
D_slices  = np.real(Bmid @ thetaD_best)

# Micromotion from rho_star using mesolve (sampled at dt=T/N purely for plotting)
rho_star_q = qt.Qobj(mat(r_star_best), dims=[[2], [2]])
args_best = {"thetaJ": thetaJ_best, "thetaD": thetaD_best, "T": T_best, "M": int(M)}

n_periods = 6
steps = n_periods * N
dt = T_best / N
t_steps = np.arange(steps + 1) * dt

res_mm = qt.mesolve(H_td, rho_star_q, t_steps, c_ops=c_ops, e_ops=None, args=args_best, options=opts)
bx = np.array(qt.expect(sx_q, res_mm.states), dtype=float)
by = np.array(qt.expect(sy_q, res_mm.states), dtype=float)
bz = np.array(qt.expect(sz_q, res_mm.states), dtype=float)
pur = np.array([float((st * st).tr().real) for st in res_mm.states], dtype=float)

strobe_idx = (np.arange(0, n_periods + 1) * N).astype(int)

floquet_eigs = np.linalg.eigvals(Phi_best)

# Stroboscopic convergence (iterate Phi_best)
K_conv = 30
rng_conv = np.random.default_rng(0)
rho0 = random_density(rng_conv)
r = vec(rho0)
conv_dist = np.zeros(K_conv + 1, dtype=float)
conv_dist[0] = float(np.linalg.norm(r - r_star_best))
for k in range(K_conv):
    r = Phi_best @ r
    conv_dist[k + 1] = float(np.linalg.norm(r - r_star_best))

json_path = run_dir / (tag + ".json")
json_path.write_text(json.dumps(sanitize_for_json(out), indent=2))

npz_path = run_dir / (tag + ".npz")
np.savez(
    npz_path,
    thetaJ=thetaJ_best,
    thetaD=thetaD_best,
    T=T_best,
    sx=sx_best,              # this is mesolve-based now
    obj=float(best["obj"]),
    hist_sx=np.array(best["history"]["sx"], dtype=float),
    hist_T=np.array(best["history"]["T"], dtype=float),
    hist_obj=np.array(best["history"]["obj"], dtype=float),
    t1=t1, Jx_t=Jx_t, D_t=D_t,
    t_s=t_s, Jx_slices=Jx_slices, D_slices=D_slices,
    t_steps=t_steps, bx=bx, by=by, bz=bz, pur=pur,
    strobe_idx=strobe_idx,
    floquet_eigs=floquet_eigs,
    conv_dist=conv_dist,
    sx_star=sx_star, sy_star=sy_star, sz_star=sz_star,
    **meta,
)

print("\nDone.")
print("  Best sx (mesolve):", sx_best)
print("  Best T:", T_best)
print("Saved:")
print("  ", json_path)
print("  ", npz_path)

[restart 1/10] obj=-0.872459, sx=0.872459, T=0.142503, nit=250
[restart 2/10] obj=-0.927161, sx=0.927161, T=0.693916, nit=221
[restart 3/10] obj=-0.915335, sx=0.915335, T=1.2218, nit=90
[restart 4/10] obj=-0.911208, sx=0.911208, T=1.74361, nit=205
[restart 5/10] obj=-0.869132, sx=0.869132, T=0.143905, nit=250
[restart 6/10] obj=-0.928988, sx=0.928988, T=0.647352, nit=169
[restart 7/10] obj=-0.922148, sx=0.922148, T=0.344749, nit=217
[restart 8/10] obj=-0.908208, sx=0.908208, T=0.169683, nit=250
[restart 9/10] obj=-0.931744, sx=0.931744, T=0.558301, nit=250
[restart 10/10] obj=-0.936456, sx=0.936456, T=0.414859, nit=250

Done.
  Best sx (mesolve): 0.9364562497537263
  Best T: 0.4148592360144335
Saved:
   results\runs\2026-03-03_210048_sigmaX_JxDelta_optT_N48_M3.json
   results\runs\2026-03-03_210048_sigmaX_JxDelta_optT_N48_M3.npz
